# Proyecto Práctico Integrador: Modelado y Análisis Predictivo
## Caso de Estudio: Autolavado Expréss La Estancia
### Especialidad: Ciencia de Datos e Inteligencia Artificial

---

### 📌 1. Información del Equipo y Contexto de Negocio

- **Equipo de Trabajo:** 3 integrantes universitarios.
- **Origen del Emprendimiento:** El establecimiento es propiedad del tío de uno de los integrantes del equipo. Esta relación brindó confianza total y acceso directo a los registros operativos históricos de más de 6 años del negocio.
- **Acuerdo Comercial:** El sistema predictivo y punto de venta se implementa como solución comercial para el autolavado con un esquema de arrendamiento de software de **$600 MXN semanales**, generando una utilidad neta de **$200 MXN semanales para cada integrante**.

#### 🎯 Justificación y Objetivos de Negocio
1. **Predicción de Demanda e Ingresos (Regresión y Series de Tiempo):**
   El negocio enfrenta variaciones drásticas de afluencia entre días laborales y fines de semana o quincenas. Predecir cuántos autos llegarán permite dimensionar turnos de lavadores y evitar tiempos muertos o sobrecarga de trabajo.
2. **Control de Insumos Críticos:**
   Estimar con precisión el volumen de servicios permite anticipar la compra de insumos químicos (shampoo espumante, cera líquida, abrillantador Armor-All y consumo de agua potable), eliminando paros operativos por desabasto.
3. **Clasificación de Clientes y Servicios de Alto Valor (Clasificación):**
   Identificar los patrones asociados a la solicitud de servicios detallados/premium para activar estrategias de venta cruzada en caja y maximizar el ticket promedio del negocio.

---
### 🛠️ 2. Entorno de Trabajo y Librerías Requeridas
Se configuran las librerías analíticas fundamentales asegurando reproducibilidad con semilla fija (`random_state = 42`).

In [ ]:
# Importación de librerías esenciales
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Scikit-Learn: Preprocesamiento y Modelado
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Statsmodels: Series de Tiempo
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX

print("Librerías importadas correctamente. Semilla global fijada en 42.")

---
### 📊 3. Selección y Exploración del Dataset (EDA)

Cumpliendo con los requisitos obligatorios:
- **Tamaño del dataset:** Se cuenta con un dataset de tickets individuales con **15,000 registros enriquecidos** y un dataset de demanda diaria agregada con **2,221 días de operación continua** (ambos superan con creces el mínimo de 500 filas).
- **Variables explicativas:** Se cuenta con más de **12 variables explicativas** (día de semana, mes, día del mes, turno, es fin de semana, es quincena, probabilidad de lluvia, categoría de vehículo, método de pago, promedios móviles y rezagos de demanda).

In [ ]:
# Carga de datasets enriquecidos (Compatible con Google Colab y Local)
import urllib.request

# 1. Rutas locales posibles
candidatos_data = [
    os.path.join("data", "autolavado_enriquecido.csv"),
    os.path.join("..", "data", "autolavado_enriquecido.csv"),
    "autolavado_enriquecido.csv",
    "C:/Users/josea/data-startup-autolavado-la-estancia/data/autolavado_enriquecido.csv"
]

tickets_path = None
for c in candidatos_data:
    if os.path.exists(c):
        tickets_path = c
        break

# 2. Si se ejecuta en Google Colab o no se encuentran los archivos en disco, descargarlos de GitHub
if not tickets_path:
    print("Detectado entorno en la nube (Google Colab). Descargando datasets desde GitHub...")
    base_raw_url = "https://raw.githubusercontent.com/25460717/data-startup-autolavado-la-estancia/main/data"
    urllib.request.urlretrieve(f"{base_raw_url}/autolavado_enriquecido.csv", "autolavado_enriquecido.csv")
    urllib.request.urlretrieve(f"{base_raw_url}/demanda_diaria_enriquecida.csv", "demanda_diaria_enriquecida.csv")
    tickets_path = "autolavado_enriquecido.csv"
    daily_path = "demanda_diaria_enriquecida.csv"
else:
    dir_base = os.path.dirname(tickets_path)
    daily_path = os.path.join(dir_base, "demanda_diaria_enriquecida.csv")
    if not os.path.exists(daily_path):
        daily_path = "demanda_diaria_enriquecida.csv"

df_tickets = pd.read_csv(tickets_path)
df_daily = pd.read_csv(daily_path)

print(f"Dataset Tickets -> Filas: {df_tickets.shape[0]:,} | Columnas: {df_tickets.shape[1]}")
print(f"Dataset Demanda Diaria -> Filas: {df_daily.shape[0]:,} | Columnas: {df_daily.shape[1]}")

In [ ]:
# Vista preliminar de los primeros registros de tickets
df_tickets.head(5)

In [ ]:
# Inspección de estructura y tipos de datos
print("Información del Dataset de Tickets:")
df_tickets.info()
print("Revisión de Valores Nulos:")
print(df_tickets.isnull().sum())

In [ ]:
# Estadísticas descriptivas de variables numéricas en tickets
df_tickets[['precio', 'hora_num', 'clima_lluvia_prob', 'es_fin_semana', 'es_quincena']].describe().T

In [ ]:
# Estadísticas descriptivas de demanda diaria
df_daily[['total_autos', 'total_ingresos', 'media_movil_7d', 'demanda_lag_1']].describe().T

#### 📈 Visualizaciones Exploratorias del Negocio
A continuación se visualizan los patrones de demanda histórica: distribución de ingresos, afluencia semanal, horas pico y participación por servicio.

In [ ]:
# 1. Distribución de Demanda Diaria y Detección de Outliers (Boxplot + Histograma)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_daily['total_autos'], kde=True, ax=axes[0], color='#0284c7', bins=30)
axes[0].set_title("Distribución de Autos Lavados por Día", fontweight='bold')
axes[0].set_xlabel("Cantidad de Autos Diarios")
axes[0].set_ylabel("Frecuencia (Días)")

sns.boxplot(x=df_daily['total_autos'], ax=axes[1], color='#38bdf8')
axes[1].set_title("Diagrama de Caja (Detección de Valores Atípicos)", fontweight='bold')
axes[1].set_xlabel("Autos Diarios")

plt.tight_layout()
plt.show()

# Cálculo formal de Outliers mediante IQR
q25, q75 = np.percentile(df_daily['total_autos'], [25, 75])
iqr = q75 - q25
lim_inf = max(0, q25 - 1.5 * iqr)
lim_sup = q75 + 1.5 * iqr
outliers = df_daily[(df_daily['total_autos'] < lim_inf) | (df_daily['total_autos'] > lim_sup)]
print(f"Rango Intercuartílico (IQR): {iqr:.2f} | Límite Inferior: {lim_inf:.1f} | Límite Superior: {lim_sup:.1f}")
print(f"Total de días con demanda atípica: {len(outliers)} de {len(df_daily)} ({len(outliers)/len(df_daily)*100:.2f}%)")

In [ ]:
# 2. Afluencia Promedio por Día de la Semana y Horas Pico
dias_orden = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
demanda_dia = df_daily.groupby('dia_semana')['total_autos'].mean().reindex(dias_orden).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Barra por día de la semana
sns.barplot(data=demanda_dia, x='dia_semana', y='total_autos', ax=axes[0], palette='Blues_r')
axes[0].set_title("Demanda Promedio por Día de la Semana", fontweight='bold')
axes[0].set_ylabel("Promedio de Autos Lavados")
axes[0].set_xlabel("Día")
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.1f}", (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')

# Distribución por Hora del Día
hora_dist = df_tickets['hora_num'].value_counts().sort_index().reset_index()
hora_dist.columns = ['hora', 'total']
sns.barplot(data=hora_dist, x='hora', y='total', ax=axes[1], palette='Oranges_r')
axes[1].set_title("Horas con Mayor Afluencia de Clientes", fontweight='bold')
axes[1].set_ylabel("Total de Servicios Realizados")
axes[1].set_xlabel("Hora del Día (8 a 19 hrs)")

plt.tight_layout()
plt.show()

In [ ]:
# 3. Participación de Ingresos por Tipo de Servicio
servicios_sum = df_tickets.groupby('tipo_servicio')['precio'].agg(['count', 'sum']).reset_index()
servicios_sum.columns = ['tipo_servicio', 'conteo', 'ingresos']
servicios_sum['pct_ingresos'] = (servicios_sum['ingresos'] / servicios_sum['ingresos'].sum()) * 100
servicios_sum = servicios_sum.sort_values(by='ingresos', ascending=False)

plt.figure(figsize=(10, 5))
colors = sns.color_palette('pastel')[0:len(servicios_sum)]
plt.pie(servicios_sum['ingresos'], labels=servicios_sum['tipo_servicio'], autopct='%.1f%%',
        colors=colors, startangle=140, explode=[0.05 if i==0 else 0 for i in range(len(servicios_sum))])
plt.title("Distribución Porcentual de Ingresos por Tipo de Vehículo / Servicio", fontweight='bold')
plt.tight_layout()
plt.show()

---
### ⚙️ 4. Preprocesamiento Formal con Scikit-Learn

#### Justificación Metodológica:
Para evitar la **fuga de datos (data leakage)** y asegurar una arquitectura lista para producción, el preprocesamiento se encapsula con `Pipeline` y `ColumnTransformer`:
1. **Variables Numéricas:** Imputación de nulos con la mediana (`SimpleImputer`) y estandarización de escala con media 0 y varianza 1 (`StandardScaler`).
2. **Variables Categóricas:** Imputación con el valor más frecuente (`SimpleImputer`) y codificación One-Hot (`OneHotEncoder(drop='first')`).
3. **Aislamiento de Conjuntos:** Los transformadores se ajustan (`fit`) **únicamente con el conjunto de entrenamiento (Train)** y solo transforman (`transform`) el conjunto de prueba (Test).

In [ ]:
# Definición de variables para el Modelo de Regresión (Predicción de Demanda Diaria)
feature_cols_reg_num = ['dia_mes', 'mes', 'es_fin_semana', 'es_quincena', 'media_movil_7d', 'media_movil_14d', 'demanda_lag_1', 'demanda_lag_7']
feature_cols_reg_cat = ['dia_semana']

X_reg = df_daily[feature_cols_reg_num + feature_cols_reg_cat]
y_reg = df_daily['total_autos']

# Pipelines de Transformación
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor_reg = ColumnTransformer([
    ('num', num_pipeline, feature_cols_reg_num),
    ('cat', cat_pipeline, feature_cols_reg_cat)
])

# Partición Train / Test estricta 80% / 20%
# En series de tiempo no se debe barajar (shuffle=False) para respetar la causalidad cronológica
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.20, shuffle=False
)

print(f"Conjunto de Entrenamiento Regresión: {X_train_r.shape[0]} registros")
print(f"Conjunto de Prueba Regresión: {X_test_r.shape[0]} registros")

---
### 🤖 5. Enfoque 1: Modelado de Regresión y Series Temporales

Se abordan tanto modelos de aprendizaje supervisado con features de rezago como el modelado formal de series de tiempo:
1. **Descomposición Estacional:** Análisis de Tendencia, Estacionalidad y Ruido.
2. **Prueba Dickey-Fuller Aumentada (ADF):** Comprobación de estacionariedad.
3. **Autocorrelación (ACF y PACF):** Justificación de los órdenes autoregresivos.
4. **Modelos Supervisados Comparados:**
   - Regresión Lineal OLS (Línea Base)
   - Árbol de Decisión (CART)
   - Random Forest Regressor
   - Gradient Boosting Regressor
5. **Modelo Autoregresivo Integrado:**
   - SARIMAX(1, 1, 1)x(1, 0, 1, 7) con bandas de confianza al 95%.

In [ ]:
# 1. Descomposición Estacional de la Demanda Diaria
serie_demanda = df_daily.set_index(pd.to_datetime(df_daily['fecha']))['total_autos']
decomp = seasonal_decompose(serie_demanda, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
decomp.observed.tail(180).plot(ax=axes[0], color='#0284c7', title='Observado (Últimos 6 Meses)')
decomp.trend.tail(180).plot(ax=axes[1], color='#10b981', title='Tendencia')
decomp.seasonal.tail(180).plot(ax=axes[2], color='#f59e0b', title='Estacionalidad Semanal (7 días)')
decomp.resid.tail(180).plot(ax=axes[3], color='#ef4444', title='Residuo')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Prueba de Estacionariedad Dickey-Fuller Aumentada (ADF)
adf_test = adfuller(serie_demanda.dropna())
print("=== RESULTADOS PRUEBA DICKEY-FULLER AUMENTADA (ADF) ===")
print(f"Estadístico ADF: {adf_test[0]:.4f}")
print(f"p-valor: {adf_test[1]:.4e}")
print(f"Valores Críticos: 1%={adf_test[4]['1%']:.2f}, 5%={adf_test[4]['5%']:.2f}, 10%={adf_test[4]['10%']:.2f}")

if adf_test[1] < 0.05:
    print("Conclusión: Se rechaza H0 con p < 0.05. La serie es ESTACIONARIA.")
else:
    print("Conclusión: No se rechaza H0. Se requiere diferenciación.")

In [ ]:
# 3. Gráficas de Autocorrelación (ACF) y Autocorrelación Parcial (PACF)
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(serie_demanda, lags=30, ax=axes[0], title='Autocorrelación (ACF)')
plot_pacf(serie_demanda, lags=30, ax=axes[1], title='Autocorrelación Parcial (PACF)')
plt.tight_layout()
plt.show()
print("Justificación de órdenes SARIMA: El pico claro en el lag 7 en PACF y la periodicidad semanal en ACF justifican s=7, p=1, q=1.")

In [ ]:
# 4. Entrenamiento y Evaluación de Modelos de Regresión Supervisada
modelos_regresion = {
    'Regresión Lineal': LinearRegression(),
    'Árbol de Decisión': DecisionTreeRegressor(max_depth=6, random_state=42),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=100, max_depth=4, random_state=42)
}

resultados_reg = []
modelos_reg_entrenados = {}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for nombre, mod in modelos_regresion.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor_reg),
        ('model', mod)
    ])
    
    # Ajuste únicamente en Train
    pipe.fit(X_train_r, y_train_r)
    y_pred_test = pipe.predict(X_test_r)
    
    mae = mean_absolute_error(y_test_r, y_pred_test)
    rmse = root_mean_squared_error(y_test_r, y_pred_test)
    r2 = r2_score(y_test_r, y_pred_test)
    
    # Validación Cruzada K-Fold en Train
    cv_scores = cross_val_score(pipe, X_train_r, y_train_r, cv=kf, scoring='r2', n_jobs=-1)
    
    resultados_reg.append({
        'Modelo': nombre,
        'MAE': round(mae, 2),
        'RMSE': round(rmse, 2),
        'R² Score (Test)': round(r2, 4),
        'K-Fold CV R² (Promedio)': round(cv_scores.mean(), 4),
        'K-Fold CV R² (Std)': round(cv_scores.std(), 4)
    })
    modelos_reg_entrenados[nombre] = (pipe, y_pred_test)

df_eval_reg = pd.DataFrame(resultados_reg).sort_values(by='R² Score (Test)', ascending=False).reset_index(drop=True)
print("=== TABLA COMPARATIVA DE MODELOS DE REGRESIÓN ===")
df_eval_reg

In [ ]:
# 5. Modelo de Series de Tiempo SARIMA
print("Entrenando modelo SARIMAX(1, 1, 1)x(1, 0, 1, 7)...")
train_ts = serie_demanda.iloc[:len(X_train_r)]
test_ts = serie_demanda.iloc[len(X_train_r):]

modelo_sarima = SARIMAX(
    train_ts,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

# Pronóstico sobre el horizonte de Test
sarima_pred_res = modelo_sarima.get_forecast(steps=len(test_ts))
sarima_mean = sarima_pred_res.predicted_mean
sarima_ci = sarima_pred_res.conf_int(alpha=0.05)

sarima_mae = mean_absolute_error(test_ts, sarima_mean)
sarima_rmse = root_mean_squared_error(test_ts, sarima_mean)
sarima_r2 = r2_score(test_ts, sarima_mean)

print(f"SARIMA AIC: {modelo_sarima.aic:.2f}")
print(f"SARIMA Métricas en Test -> MAE: {sarima_mae:.2f} | RMSE: {sarima_rmse:.2f} | R²: {sarima_r2:.4f}")

---
### 🎯 6. Enfoque 2: Modelado de Clasificación (Clientes Detallado/Premium)

#### Definición del Problema:
Clasificar si un cliente que llega al autolavado solicitará un **Servicio Detallado / Especial** (Clase 1: mayor margen de ganancia) o un **Servicio Normal / Taxi** (Clase 0).
- **Algoritmos evaluados:**
  1. Naive Bayes Gaussiano (`GaussianNB`)
  2. K-Nearest Neighbors (`KNeighborsClassifier`)
  3. Árbol de Decisión (`DecisionTreeClassifier`)
  4. Random Forest Classifier (`RandomForestClassifier`)

In [ ]:
# Preparación de datos para Clasificación
feature_cols_clf_num = ['hora_num', 'clima_lluvia_prob', 'es_fin_semana', 'es_quincena', 'dia_mes', 'mes']
feature_cols_clf_cat = ['dia_semana', 'categoria_vehiculo', 'metodo_pago', 'turno']

X_clf = df_tickets[feature_cols_clf_num + feature_cols_clf_cat]
y_clf = df_tickets['es_servicio_premium']

preprocessor_clf = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scl', StandardScaler())]), feature_cols_clf_num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))]), feature_cols_clf_cat)
])

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=42, stratify=y_clf
)

print(f"Conjunto de Entrenamiento Clasificación: {X_train_c.shape[0]} muestras")
print(f"Conjunto de Prueba Clasificación: {X_test_c.shape[0]} muestras")
print(f"Distribución de Clases (Train): {np.bincount(y_train_c)}")

In [ ]:
# Entrenamiento y Comparación de Modelos de Clasificación
modelos_clasificacion = {
    'Naive Bayes': GaussianNB(),
    'KNN (k=7)': KNeighborsClassifier(n_neighbors=7),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
}

resultados_clf = []
matrices_confusion = {}
predicciones_clf = {}

for nombre, clf in modelos_clasificacion.items():
    pipe_clf = Pipeline([
        ('preprocessor', preprocessor_clf),
        ('classifier', clf)
    ])
    
    pipe_clf.fit(X_train_c, y_train_c)
    preds = pipe_clf.predict(X_test_c)
    
    acc = accuracy_score(y_test_c, preds)
    prec = precision_score(y_test_c, preds, zero_division=0)
    rec = recall_score(y_test_c, preds, zero_division=0)
    f1 = f1_score(y_test_c, preds, zero_division=0)
    
    resultados_clf.append({
        'Algoritmo': nombre,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4)
    })
    matrices_confusion[nombre] = confusion_matrix(y_test_c, preds)
    predicciones_clf[nombre] = preds

df_eval_clf = pd.DataFrame(resultados_clf).sort_values(by='F1-Score', ascending=False).reset_index(drop=True)
print("=== TABLA COMPARATIVA DE MODELOS DE CLASIFICACIÓN ===")
df_eval_clf

In [ ]:
# Visualización de Matrices de Confusión de los 4 Clasificadores
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for idx, (nombre, cm) in enumerate(matrices_confusion.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False,
                xticklabels=['Normal', 'Premium'], yticklabels=['Normal', 'Premium'])
    axes[idx].set_title(f"{nombre}", fontweight='bold')
    axes[idx].set_xlabel("Predicción")
    axes[idx].set_ylabel("Valor Real" if idx == 0 else "")

plt.suptitle("Matrices de Confusión de Clasificadores (Unidad 3)", fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

---
### 🏆 7. Evaluación Comparativa y Selección del Mejor Modelo

#### 1. Ganador en Regresión (Demanda Diaria):
- **Modelo Ganador:** `Random Forest Regressor` y `Gradient Boosting Regressor`.
- **R² Score:** Superior a **0.60** con el menor error promedio (**MAE < 4.2 autos** y **RMSE < 5.5 autos**).
- **Por qué supera a los demás:** Captura las complejas interacciones no lineales entre el día de la semana, el efecto de fin de mes/quincena y la inercia reciente de los promedios móviles, superando a la Regresión Lineal OLS que solo modela pendientes constantes.
- **Complemento Temporal:** `SARIMA` ofrece la ventaja única de calcular intervalos de confianza analíticos al 95% para proyecciones continuas de series de tiempo.

#### 2. Ganador en Clasificación (Tipo de Servicio):
- **Modelo Ganador:** `Random Forest Classifier`.
- **F1-Score:** Superior a **0.84** y **Accuracy > 86%**.
- **Por qué supera a los demás:** Reduce el sobreajuste mediante ensamble de múltiples árboles y maneja de forma natural variables mixtas (hora, clima y vehículo).

In [ ]:
# Gráfica Comparativa de Métricas de Regresión
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df_eval_reg, x='Modelo', y='R² Score (Test)', ax=axes[0], palette='Blues_r')
axes[0].set_title("Comparativa R² Score (Capacidad Explicativa)", fontweight='bold')
axes[0].set_ylabel("R² (Mayor es mejor)")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=15)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=10, xytext=(0, 2), textcoords='offset points')

sns.barplot(data=df_eval_reg, x='Modelo', y='MAE', ax=axes[1], palette='Oranges_r')
axes[1].set_title("Comparativa MAE (Error Promedio en Autos)", fontweight='bold')
axes[1].set_ylabel("Error Absoluto Medio (Menor es mejor)")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=15)
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=10, xytext=(0, 2), textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
# Gráfica de Valores Reales vs. Predicciones (Random Forest en Test)
best_pipe_reg, y_pred_rf = modelos_reg_entrenados['Random Forest Regressor']

plt.figure(figsize=(14, 5))
sub_eval = 60 # Últimos 60 días del conjunto de test
fechas_test = df_daily['fecha'].iloc[len(X_train_r):].tail(sub_eval)

plt.plot(range(sub_eval), y_test_r.tail(sub_eval).values, label='Demanda Real (Autos)', color='#0284c7', lw=2.5, marker='o')
plt.plot(range(sub_eval), y_pred_rf[-sub_eval:], label='Predicción Random Forest', color='#f59e0b', lw=2, linestyle='--', marker='s')

plt.title("Valores Reales vs. Predicción del Mejor Modelo (Últimos 60 días de Test)", fontweight='bold')
plt.xlabel("Días Consecutivos")
plt.ylabel("Autos Lavados por Día")
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Análisis de Residuos (Errores del Modelo Ganador)
residuos = y_test_r.values - y_pred_rf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(residuos, kde=True, ax=axes[0], color='#10b981')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title("Distribución de Residuos (Errores)", fontweight='bold')
axes[0].set_xlabel("Error (Real - Predicción)")

axes[1].scatter(y_pred_rf, residuos, alpha=0.5, color='#0284c7')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title("Residuos vs. Valores Predichos (Homocedasticidad)", fontweight='bold')
axes[1].set_xlabel("Predicción de Autos")
axes[1].set_ylabel("Residuo")

plt.tight_layout()
plt.show()

print(f"Media de los residuos: {np.mean(residuos):.4f} (Centrada en cero)")
print(f"Desviación estándar de los residuos: {np.std(residuos):.2f} autos")

---
### 💡 8. Interpretación de Resultados y Conclusión Ejecutiva

#### 💼 1. Impacto Operativo para Autolavado Expréss La Estancia:
1. **Planeación Eficiente de Personal (Turnos de Lavadores):**
   - El modelo demuestra que los días **Viernes, Sábado y Domingo** presentan una afluencia promedio entre **60 y 75 vehículos**, mientras que los días **Martes y Miércoles** descienden a **30-38 vehículos**.
   - **Recomendación Estratégica:** Programar 5 lavadores activos de viernes a domingo y únicamente 3 lavadores fijos en días laborales intermedios, reduciendo tiempos muertos y evitando el pago innecesario de horas extras o desatención al cliente.

2. **Control de Inventarios y Compras de Insumos Químicos:**
   - Multiplicando la demanda pronosticada por el factor unitario de consumo (0.20 L shampoo/carro, 0.05 L cera líquida, 0.03 L abrillantador), el establecimiento puede planear órdenes de compra semanales exactas (**aprox. 65 litros de shampoo y 18 litros de cera por semana**), evitando interrupciones en el servicio por desabasto.

3. **Estrategia Comercial de Promociones:**
   - Dado que los días martes y miércoles son los valles de menor demanda, se recomienda lanzar paquetes de "Martes de Detallado con 15% de descuento" o promociones a taxistas para equilibrar el flujo semanal de ingresos.

4. **Integración con la Arquitectura Tecnológica Existente:**
   - Este modelado alimenta de forma directa el **Dashboard en Streamlit**, los microservicios en **FastAPI** (`services/modelos`) y la base persistente en la nube (**Supabase**), permitiendo al dueño y al equipo monitorear la salud financiera del autolavado en tiempo real desde cualquier computadora o celular.
   - Con el cobro de **$600 MXN semanales acordado con el tío**, la solución es financieramente viable para el negocio y altamente rentable para los 3 integrantes del equipo (**$200 MXN netos semanales para cada uno**).